# 📈 ScanTrade - ETF Strategy Search & Backtesting Engine

Questo Jupyter Notebook implementa il sistema **ScanTrade** per la ricerca, l'ottimizzazione e il backtesting di strategie operative di trading su ETF.

### 🧩 Moduli del Sistema:
1. **Data Engine**: Retrieval dati storici ETF e calcolo indicatori tecnici (SMA, EMA, RSI, MACD, ATR).
2. **Strategy Engine**: Generazione segnali di ingresso/uscita e gestione del rischio (Stop Loss & Take Profit).
3. **Backtest Engine**: Simulazione storica barra-per-barra con capitale iniziale, commissioni e slippage.
4. **Strategy Finder**: Algoritmo di Grid Search per la ricerca ed ottimizzazione dei parametri.
5. **Analytics Engine & Reporter**: Calcolo metriche finanziarie (ROI, CAGR, Sharpe, Sortino, Max Drawdown, Win Rate, Profit Factor) e dashboard grafica.

In [ ]:
# Import delle librerie necessarie
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools
import warnings
warnings.filterwarnings('ignore')

## 1. Data Engine
Gestisce l'acquisizione dei dati storici degli ETF (tramite Yahoo Finance o generatore sintetico di fallback) e il calcolo degli indicatori tecnici principali.

In [ ]:
class DataEngine:
    @staticmethod
    def generate_synthetic_etf_data(days=1000, start_price=100.0, seed=42):
        np.random.seed(seed)
        dates = pd.date_range(end=pd.Timestamp.today(), periods=days, freq='B')
        returns = np.random.normal(0.0004, 0.015, days)
        price = start_price * np.exp(np.cumsum(returns))
        
        high = price * (1 + np.abs(np.random.normal(0, 0.008, days)))
        low = price * (1 - np.abs(np.random.normal(0, 0.008, days)))
        open_p = low + (high - low) * np.random.uniform(0.2, 0.8, days)
        close = price
        volume = np.random.randint(100000, 5000000, days)
        
        df = pd.DataFrame({
            'Open': open_p,
            'High': high,
            'Low': low,
            'Close': close,
            'Volume': volume
        }, index=dates)
        return df

    @staticmethod
    def fetch_etf_data(symbol="SPY", period="3y"):
        try:
            import yfinance as yf
            df = yf.download(symbol, period=period, progress=False)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            if df.empty:
                raise ValueError("Nessun dato scaricato.")
            return df
        except Exception as e:
            print(f"⚠️ Download diretto yfinance non disponibile ({e}). Generazione dati sintetici per {symbol}...")
            return DataEngine.generate_synthetic_etf_data()

    @staticmethod
    def add_indicators(df, fast_sma=20, slow_sma=50, rsi_period=14, atr_period=14):
        df = df.copy()
        df['SMA_Fast'] = df['Close'].rolling(window=fast_sma).mean()
        df['SMA_Slow'] = df['Close'].rolling(window=slow_sma).mean()
        df['EMA_Fast'] = df['Close'].ewm(span=fast_sma, adjust=False).mean()
        df['EMA_Slow'] = df['Close'].ewm(span=slow_sma, adjust=False).mean()
        
        delta = df['Close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=rsi_period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=rsi_period).mean()
        rs = gain / (loss + 1e-10)
        df['RSI'] = 100 - (100 / (1 + rs))
        
        ema12 = df['Close'].ewm(span=12, adjust=False).mean()
        ema26 = df['Close'].ewm(span=26, adjust=False).mean()
        df['MACD'] = ema12 - ema26
        df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
        
        high_low = df['High'] - df['Low']
        high_close = (df['High'] - df['Close'].shift()).abs()
        low_close = (df['Low'] - df['Close'].shift()).abs()
        tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        df['ATR'] = tr.rolling(window=atr_period).mean()
        return df

## 2. Strategy Engine
Definisce la logica operativa di compravendita e i parametri di gestione del rischio (Stop Loss & Take Profit).

In [ ]:
class StrategyEngine:
    def __init__(self, fast_window=20, slow_window=50, rsi_sell=70, stop_loss_pct=0.03, take_profit_pct=0.08):
        self.fast_window = fast_window
        self.slow_window = slow_window
        self.rsi_sell = rsi_sell
        self.stop_loss_pct = stop_loss_pct
        self.take_profit_pct = take_profit_pct

    def generate_signals(self, df):
        data = DataEngine.add_indicators(
            df,
            fast_sma=self.fast_window,
            slow_sma=self.slow_window
        ).copy()
        
        data['Signal'] = 0
        trend_bull = data['SMA_Fast'] > data['SMA_Slow']
        trend_bear = data['SMA_Fast'] < data['SMA_Slow']
        
        buy_condition = trend_bull & (data['RSI'] > 45) & (data['RSI'] < self.rsi_sell)
        sell_condition = trend_bear | (data['RSI'] > self.rsi_sell)
        
        data.loc[buy_condition, 'Signal'] = 1
        data.loc[sell_condition, 'Signal'] = -1
        return data

## 3. Backtest Engine
Esegue la simulazione storica barra per barra tracciando capitale, commissioni di transazione e slippage.

In [ ]:
class BacktestEngine:
    def __init__(self, initial_capital=10000.0, commission_pct=0.001, slippage_pct=0.0005):
        self.initial_capital = initial_capital
        self.commission_pct = commission_pct
        self.slippage_pct = slippage_pct

    def run(self, df, strategy):
        data = strategy.generate_signals(df)
        capital = self.initial_capital
        position = 0.0
        entry_price = 0.0
        entry_date = None
        
        equity_curve = []
        trades = []
        
        for i in range(len(data)):
            date = data.index[i]
            close = float(data['Close'].iloc[i])
            high = float(data['High'].iloc[i])
            low = float(data['Low'].iloc[i])
            signal = int(data['Signal'].iloc[i])
            
            # Gestione posizione aperta (Check Stop Loss e Take Profit)
            if position > 0:
                sl_price = entry_price * (1 - strategy.stop_loss_pct)
                tp_price = entry_price * (1 + strategy.take_profit_pct)
                
                exit_price = None
                exit_reason = None
                
                if low <= sl_price:
                    exit_price = sl_price * (1 - self.slippage_pct)
                    exit_reason = "Stop Loss"
                elif high >= tp_price:
                    exit_price = tp_price * (1 - self.slippage_pct)
                    exit_reason = "Take Profit"
                elif signal == -1:
                    exit_price = close * (1 - self.slippage_pct)
                    exit_reason = "Signal Exit"
                    
                if exit_price is not None:
                    proceeds = position * exit_price
                    commission = proceeds * self.commission_pct
                    capital = proceeds - commission
                    pnl = (exit_price - entry_price) / entry_price
                    trades.append({
                        'Entry Date': entry_date,
                        'Exit Date': date,
                        'Entry Price': entry_price,
                        'Exit Price': exit_price,
                        'PnL (%)': pnl * 100,
                        'Reason': exit_reason
                    })
                    position = 0.0
                    entry_price = 0.0
                    entry_date = None

            # Apertura nuova posizione
            if position == 0 and signal == 1:
                buy_price = close * (1 + self.slippage_pct)
                commission = capital * self.commission_pct
                investable = capital - commission
                position = investable / buy_price
                entry_price = buy_price
                entry_date = date
                capital = 0.0

            # Calcolo valore corrente dell'equity
            current_val = capital if position == 0 else position * close
            equity_curve.append(current_val)

        equity_series = pd.Series(equity_curve, index=data.index)
        trades_df = pd.DataFrame(trades)
        return equity_series, trades_df, data

## 4. Analytics & Reporting Engine
Calcola le metriche chiave di rendimento e rischio e visualizza il dashboard prestazionale.

In [ ]:
class AnalyticsEngine:
    @staticmethod
    def calculate_metrics(equity_series, trades_df, initial_capital=10000.0):
        if equity_series.empty or len(equity_series) < 2:
            return {}
            
        total_return = (equity_series.iloc[-1] - initial_capital) / initial_capital * 100
        days = (equity_series.index[-1] - equity_series.index[0]).days
        years = max(days / 365.25, 0.01)
        cagr = (((equity_series.iloc[-1] / initial_capital) ** (1 / years)) - 1) * 100
        
        daily_returns = equity_series.pct_change().dropna()
        sharpe_ratio = (daily_returns.mean() / (daily_returns.std() + 1e-10)) * np.sqrt(252)
        
        downside_returns = daily_returns[daily_returns < 0]
        sortino_ratio = (daily_returns.mean() / (downside_returns.std() + 1e-10)) * np.sqrt(252)
        
        cummax = equity_series.cummax()
        drawdowns = (equity_series - cummax) / cummax * 100
        max_drawdown = drawdowns.min()
        
        num_trades = len(trades_df)
        if num_trades > 0:
            win_trades = trades_df[trades_df['PnL (%)'] > 0]
            win_rate = len(win_trades) / num_trades * 100
            
            gross_profit = trades_df[trades_df['PnL (%)'] > 0]['PnL (%)'].sum()
            gross_loss = abs(trades_df[trades_df['PnL (%)'] < 0]['PnL (%)'].sum())
            profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.nan
        else:
            win_rate = 0.0
            profit_factor = 0.0
            
        return {
            'Total Return (%)': round(total_return, 2),
            'CAGR (%)': round(cagr, 2),
            'Sharpe Ratio': round(sharpe_ratio, 2),
            'Sortino Ratio': round(sortino_ratio, 2),
            'Max Drawdown (%)': round(max_drawdown, 2),
            'Win Rate (%)': round(win_rate, 2),
            'Profit Factor': round(profit_factor, 2) if not np.isnan(profit_factor) else "N/A",
            'Total Trades': num_trades
        }

    @staticmethod
    def plot_dashboard(df, equity_series, trades_df, title="ScanTrade Performance Dashboard"):
        plt.figure(figsize=(14, 10))
        
        # Subplot 1: Price & Buy/Sell signals
        ax1 = plt.subplot(3, 1, 1)
        ax1.plot(df.index, df['Close'], label='ETF Close Price', color='black', alpha=0.75)
        if 'SMA_Fast' in df.columns:
            ax1.plot(df.index, df['SMA_Fast'], label='SMA Fast', color='blue', linestyle='--')
        if 'SMA_Slow' in df.columns:
            ax1.plot(df.index, df['SMA_Slow'], label='SMA Slow', color='orange', linestyle='--')
            
        if not trades_df.empty:
            ax1.scatter(trades_df['Entry Date'], trades_df['Entry Price'], marker='^', color='green', s=80, label='Buy', zorder=5)
            ax1.scatter(trades_df['Exit Date'], trades_df['Exit Price'], marker='v', color='red', s=80, label='Sell', zorder=5)
            
        ax1.set_title(f"{title} - Quotazione ETF e Trade", fontsize=12, fontweight='bold')
        ax1.set_ylabel("Prezzo ($)")
        ax1.legend(loc='upper left')
        ax1.grid(True, alpha=0.3)
        
        # Subplot 2: Equity Curve vs Buy & Hold
        ax2 = plt.subplot(3, 1, 2, sharex=ax1)
        buy_hold = (df['Close'] / df['Close'].iloc[0]) * equity_series.iloc[0]
        ax2.plot(equity_series.index, equity_series, label='ScanTrade Strategy Equity', color='green', linewidth=2)
        ax2.plot(buy_hold.index, buy_hold, label='Benchmark Buy & Hold', color='gray', linestyle=':', linewidth=1.5)
        ax2.set_title("Evoluzione del Capitale ($)", fontsize=12, fontweight='bold')
        ax2.set_ylabel("Valore Portafoglio ($)")
        ax2.legend(loc='upper left')
        ax2.grid(True, alpha=0.3)
        
        # Subplot 3: Drawdown
        ax3 = plt.subplot(3, 1, 3, sharex=ax1)
        cummax = equity_series.cummax()
        drawdown = (equity_series - cummax) / cummax * 100
        ax3.fill_between(drawdown.index, drawdown, 0, color='red', alpha=0.3, label='Drawdown %')
        ax3.set_title("Profilo di Drawdown (%)", fontsize=12, fontweight='bold')
        ax3.set_ylabel("Drawdown (%)")
        ax3.legend(loc='lower left')
        ax3.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()

## 5. Strategy Finder (Grid Search Optimization)
Esegue un'esplorazione automatica nello spazio dei parametri per individuare la combinazione che massimizza il profilo rendimento/rischio (Sharpe Ratio).

In [ ]:
class StrategyFinder:
    def __init__(self, df, backtester=None):
        self.df = df
        self.backtester = backtester or BacktestEngine()

    def grid_search(self, fast_range=[10, 20, 30], slow_range=[40, 60, 100], sl_range=[0.02, 0.04], tp_range=[0.06, 0.10]):
        results = []
        combinations = list(itertools.product(fast_range, slow_range, sl_range, tp_range))
        print(f"🔍 Test di {len(combinations)} combinazioni di parametri in corso...")
        
        for fast, slow, sl, tp in combinations:
            if fast >= slow:
                continue
            strategy = StrategyEngine(fast_window=fast, slow_window=slow, stop_loss_pct=sl, take_profit_pct=tp)
            equity, trades, _ = self.backtester.run(self.df, strategy)
            metrics = AnalyticsEngine.calculate_metrics(equity, trades, self.backtester.initial_capital)
            
            if metrics:
                results.append({
                    'Fast Window': fast,
                    'Slow Window': slow,
                    'Stop Loss': sl,
                    'Take Profit': tp,
                    'Total Return (%)': metrics['Total Return (%)'],
                    'Sharpe Ratio': metrics['Sharpe Ratio'],
                    'Max Drawdown (%)': metrics['Max Drawdown (%)'],
                    'Win Rate (%)': metrics['Win Rate (%)'],
                    'Trades': metrics['Total Trades']
                })
                
        results_df = pd.DataFrame(results)
        if not results_df.empty:
            results_df = results_df.sort_values(by='Sharpe Ratio', ascending=False).reset_index(drop=True)
        return results_df

## 6. Esecuzione Completa della Simulazione
Eseguiamo il flusso completo: caricamento dati ETF, ricerca della miglior configurazione tramite Grid Search, e visualizzazione dei risultati finali.

In [ ]:
# 1. Download/Generazione Dati ETF (es. SPY)
symbol = "SPY"
print(f"📥 Recupero dati storici per l'ETF: {symbol}...")
df_etf = DataEngine.fetch_etf_data(symbol=symbol, period="3y")
print(f"✅ Dati caricati: {len(df_etf)} barre dal {df_etf.index[0].strftime('%Y-%m-%d')} al {df_etf.index[-1].strftime('%Y-%m-%d')}")

# 2. Ricerca della Strategia Ottimale tramite Grid Search
finder = StrategyFinder(df_etf)
grid_results = finder.grid_search(
    fast_range=[10, 20, 30],
    slow_range=[40, 60, 90],
    sl_range=[0.03, 0.05],
    tp_range=[0.08, 0.12]
)

print("\n🏆 TOP 5 STRATEGIE INDIVIDUATE:")
display(grid_results.head(5))

# 3. Simulation & Dashboard sulla migliore strategia trovata
if not grid_results.empty:
    best = grid_results.iloc[0]
    best_strategy = StrategyEngine(
        fast_window=int(best['Fast Window']),
        slow_window=int(best['Slow Window']),
        stop_loss_pct=best['Stop Loss'],
        take_profit_pct=best['Take Profit']
    )
    
    backtester = BacktestEngine(initial_capital=10000.0, commission_pct=0.001)
    equity, trades, df_signals = backtester.run(df_etf, best_strategy)
    
    metrics = AnalyticsEngine.calculate_metrics(equity, trades)
    print("\n📊 METRICHE PRESTAZIONALI STRATEGIA OTTIMALE:")
    for metric_name, value in metrics.items():
        print(f" - {metric_name:<20}: {value}")
        
    AnalyticsEngine.plot_dashboard(
        df_signals, 
        equity, 
        trades, 
        title=f"ScanTrade - ETF {symbol} (SMA Fast={int(best['Fast Window'])}, Slow={int(best['Slow Window'])})"
    )